In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["KERAS_BACKEND"] = "torch"
import keras
keras.mixed_precision.set_global_policy("mixed_float16")
keras.config.set_image_data_format("channels_first")
from pathlib import Path
import torch

from PIL import Image
import numpy as np
# Local files/code
from src.agents.U_NET import UNET
import cv2
import matplotlib.pyplot as plt
import src.config as config
from pathlib import Path
from src.data_preprocessing.segmentation import get_percent_of_Disease
import pandas as pd
import src.data_preprocessing.image_preprocessing as image_pre

In [ ]:
#load in the best Segmentation model
best = keras.models.load_model(r"./models/unet_models/UNet_Best.keras")

In [ ]:
#preprocess the image before passing to the model for prediction
def prep_image(path, size=(384,384)):
    #convert the image to a numpy array
    img = np.asarray(Image.open(path).convert('RGB'))
    #utilize image_pre to crop any borders (if they exist)
    img = image_pre.crop_border(img)

    #resize the image according to the image size the UNet takes
    #utilize cv2's reccommended pixel interpolation for image shrinking (as most GBIF images are 500x500)
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)

    #collapse the pixel values into a range of 0-1
    x = img.astype(np.float32) / 255
    return img, x

In [ ]:
#create the path to the Preprocessed GBIF Images
GBIF_IMAGES = config.Data.GBIF.IMAGES
#create a list containing both folder paths for All GBIF images and the PlantVQA images
folders = list(GBIF_IMAGES.iterdir()) + [config.Data.PlantExpertVQA.ROOT / "images"]
#create a segmented_images folder to store the masks (if it does not exist)
Path.mkdir(config.Data.SEGMENTED_IMAGES, exist_ok=True)
for folder in folders:
    #generate a corresponding mask folder for the image folder
    Mask_Folder = config.Data.SEGMENTED_IMAGES / folder.name
    Path.mkdir(Mask_Folder, exist_ok=True)
    #loop through the images within the folder
    for item in folder.iterdir():
        #create a mask path that will have the same name as the image, only difference is extension
        #This allows us to create a trusted unique connection between each image and mask
        mask_path = Mask_Folder / str(item.stem + '.png')
        if Path.exists(mask_path):
            continue
        
        #perform needed preprocessing on the images
        img, x = prep_image(item)
        
        #use the best model to predict a mask
        mask = best.predict(x, verbose=0)
        #Set the threshold for a value of confidence we want our probabilities to reach to be considered a disease pixel
        #mulitply by 255 to create a fully white pixel for diseased pixels
        mask = (mask > 0.7).astype(np.uint8) * 255
        #remove the extra tensorflow batch dimension
        mask = mask.squeeze()

        #save the mask
        if not cv2.imwrite(str(mask_path), mask):
            print('failed to save mask', mask_path)
    


In [ ]:
#clean path takes a path and breaks it 
def clean_path(p: Path) -> str:
    parts = p.resolve().parts
    i = parts.index('data')
    return Path(*parts[i:]).as_posix()

In [ ]:
import importlib, src.data_preprocessing.segmentation as seg
importlib.reload(seg)
get_percent_of_Disease = seg.get_percent_of_Disease

#load the directory of Masked Images
MASK_DIR = config.Data.SEGMENTED_IMAGES
gbif_df = []

for folder in GBIF_IMAGES.iterdir():
    #grab the corresponding mask folder
    Mask_Folder = config.Data.SEGMENTED_IMAGES / folder.name
    for item in folder.iterdir():
        #the mask name will match the image name, just alter prefix
        mask_path = Mask_Folder / str(item.stem + '.png')
        if not mask_path.exists():
            continue

        #calculate percentage of disease present within the plant
        dis, sev = get_percent_of_Disease(item, mask_path)
        #build a dictionary object corresponding of all important data for classifier training
        plant_dict = {'symbol' : folder.name,
                      'img_id' : item.stem,
                      'img_path': clean_path(item),
                      'mask_path': clean_path(mask_path),
                      'disease' : dis,
                      'severity' : sev}
        gbif_df.append(plant_dict)
#build the final DF of updated data
final_df = pd.DataFrame(gbif_df)
final_df.to_csv('GBIF_UPDATED_SEVERITY.csv', index=False)